# Практика. Перевод текстов на примере WMT20

В предыдущих уроках вы узнали про encoder-decoder модели, seq2seq-задачи и метрики для них. Теперь перенесём всё это на прикладную задачу — машинный перевод. Вы шаг за шагом пройдёте весь путь: загрузите корпус WMT20, обучите модель и оцените её по BLEU, chrF.

## 1. Данные
Прежде чем запускать модели, важно понять, что мы подаём на вход. В WMT20 пары предложений соответствуют текстам в оригинале и переводе. Мы будем работать с направлением en→ru. Посмотрим на [англо-русский параллельный датасет](https://huggingface.co/datasets/yezhengli9/wmt20-en-ru). В нём есть два поля: `id` и `translation`, второе состоит из полей `en` — оригинал на английском, `ru` — перевод на русский. 

Сначала приведём корпус к единому формату `{"src": <оригинал>, "tgt": <перевод>}` и добавим базовую проверку качества — удалим чрезмерно длинные строки.

### Задание 1
Загрузите датасет через библиотеку `datasets`, разделите поле `translation` на поля `src` и `tgt` и удалите строки длиной больше 1000 символов. В конце отделите 20% выборки в тестовую часть с помощью метода `.train_test_split()`.

In [3]:
from datasets import load_dataset, Dataset, DatasetDict
import ast

def format_row(row):
    translation = ast.literal_eval(row["translation (translation)"])
    return {
        "id": row["id (string)"],
        "src": translation["en"],
        "tgt": translation["ru"],
    }
    
def filter_rows(row, max_len: int = 1000):
    return (
        row["src"] is not None
        and row["tgt"] is not None
        and len(row["src"]) <= max_len
        and len(row["tgt"]) <= max_len
    )


def load_wmt(max_len: int = 1000, test_size: float = 0.20, seed: int = 42):
    ds = load_dataset("yezhengli9/wmt20-en-ru")["train"]
    ds = ds.map(format_row, remove_columns=["translation (translation)", "id (string)"])
    ds = ds.filter(lambda row: filter_rows(row, max_len))
    ds = ds.train_test_split(test_size=test_size, seed=seed)
    return ds

ds = load_wmt()

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'src', 'tgt'],
        num_rows: 1601
    })
    test: Dataset({
        features: ['id', 'src', 'tgt'],
        num_rows: 401
    })
})

In [5]:
ds["train"][:3]

{'id': ['1043', '1533', '880'],
 'src': ['The Justice Department later said it was opening a probe of online platforms.\n',
  'French investigating magistrates must now determine whether the Sudanese complaint is admissible and whether to open an investigation.\n',
  '"Shaq is not ready yet, he\'s doing rehab stuff," said Klopp.\n'],
 'tgt': ['Позже в Министерстве юстиции заявили, что начнут следить за деятельностью онлайн-платформ.\n',
  'Теперь французские следственные судьи должны определить, допустимо ли суданское заявление и будет ли начато расследование.\n',
  '“Шак еще не готов, он на реабилитации”, - сказал Клопп.\n']}

## 2. Метрики: BLEU, chrF

Метрики для seq2seq-задач можно реализовать вручную, а можно использовать быстро из библиотек, например, `evaluate`. Посчитать метрику chrF для наших примеров можно так:



In [6]:
import evaluate
chrf = evaluate.load('chrf')
src = [
    "No evidence of acute myocardial infarction.",
    "The patient was prescribed 5 mg of warfarin daily.",
    "CT scan revealed a 2 cm lesion in the left lobe of the liver."
]
ref = [
    "Признаков острого инфаркта миокарда не выявлено.",
    "Пациенту назначено 5 мг варфарина ежедневно.",
    "КТ выявила очаг размером 2 см в левой доле печени."
]
hyp = [
    "Нет признаков острого инфаркта миокарда.",        
    "Пациенту назначено 5 мг варфарина каждый день.",   
    "КТ показала очаг 2 мм в левой доле печени."       
]

for i in range(len(src)):
    print(f"SRC: {src[i]}")
    print(f"REF: {ref[i]}")
    print(f"HYP: {hyp[i]}")
    print(f"chrF: {chrf.compute(predictions=[hyp[i]], references=[ref[i]])}")
    print("---\n")

SRC: No evidence of acute myocardial infarction.
REF: Признаков острого инфаркта миокарда не выявлено.
HYP: Нет признаков острого инфаркта миокарда.
chrF: {'score': 73.64024156814074, 'char_order': 6, 'word_order': 0, 'beta': 2}
---

SRC: The patient was prescribed 5 mg of warfarin daily.
REF: Пациенту назначено 5 мг варфарина ежедневно.
HYP: Пациенту назначено 5 мг варфарина каждый день.
chrF: {'score': 74.27254251887318, 'char_order': 6, 'word_order': 0, 'beta': 2}
---

SRC: CT scan revealed a 2 cm lesion in the left lobe of the liver.
REF: КТ выявила очаг размером 2 см в левой доле печени.
HYP: КТ показала очаг 2 мм в левой доле печени.
chrF: {'score': 54.70692668956404, 'char_order': 6, 'word_order': 0, 'beta': 2}
---



### Задание 2
Напишите функцию, которая считает BLEU и chrF на корпусе. Используйте библиотеку `evaluate`.

In [7]:
import evaluate


def compute_bleu_chrf(hyp: list[str], ref: list[str]):
    bleu = evaluate.load("bleu")
    chrf = evaluate.load("chrf")
    bleu = bleu.compute(predictions=hyp, references=ref)
    chrf = chrf.compute(predictions=hyp, references=ref)
    return {"bleu": bleu['bleu'], "chrf": chrf['score']} 


compute_bleu_chrf(ds["train"]["tgt"][:], ds["train"]["src"][:])

{'bleu': 0.011779417463131894, 'chrf': 2.380412850102792}

## 3. Бейзлайн без дообучения

Любое решение имеет смысл сравнивать с каким-то простым. 

Возьмём [opus-mt-en-ru](https://huggingface.co/Helsinki-NLP/opus-mt-en-ru) — маленькую предобученную модель перевода с английского на русский. Проверим, как она переводит предложения «из коробки». Для этого подадим в неё промпт, как и в LLM. Этот бейзлайн послужит нам отправной точкой, с которой сравним качество нашей модели после дообучения (по метрикам BLEU и chrF).

### Задание 3

В коде уже есть код для загрузки модели и токенизатора, а также для замера метрик на части теста. 

Допишите функцию seq2seq-генерации, которая получит список текстов, запустит батчевую генерацию и вернёт список сгенерированных переводов.

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
mdl = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-ru").to("cuda")

@torch.no_grad()
def translate_batch(texts: list[str], max_new_tokens: int = 128) -> list[str]:
    inputs = tok(texts,
                 return_tensors="pt",
                 padding=True,
                 truncation=True)
    mdl.eval()
    outputs = mdl.generate(
        inputs.input_ids.to("cuda"),
        max_new_tokens=max_new_tokens,
    )
    return tok.batch_decode(outputs, skip_special_tokens=True)

N = 100 # можно изменять в зависимости от доступных ресурсов
val_src = [r["src"] for r in ds["test"]][:N]
val_ref = [r["tgt"] for r in ds["test"]][:N]

hyp = translate_batch(val_src, max_new_tokens=128)

metrics = compute_bleu_chrf(hyp, val_ref)
print({k: round(v, 2) for k, v in metrics.items()})

{'bleu': 0.2, 'chrf': 49.89}


In [9]:
for i in range(N):
    print(f"SRC: {val_src[i]}")
    print(f"REF: {val_ref[i]}")
    print(f"HYP: {hyp[i]}")
    print("---\n")

SRC: Taoiseach opens new Irish Consulate in LA

REF: Премьер-министр открыл новое консульство Ирландии в Лос-Анджелесе

HYP: Тайосич открывает новое ирландское консульство в Лос-Анджелесе
---

SRC: He's been superb.

REF: Он был великолепен.

HYP: Он был великолепен.
---

SRC: "Rising population has become like a second stage cancer.

REF: “Рост населения стал похож на рак второй стадии.

HYP: &quot; Растущее население стало похоже на рак второго этапа.
---

SRC: Perhaps the thing Cramer likes most about the two companies?

REF: Что в этих двух компаниях Крамеру нравится, наверное, больше всего?

HYP: Возможно, то, что Крамеру больше всего нравится в этих двух компаниях?
---

SRC: The two were booked at Palmdale Sheriff's Station and a search warrant was served at their residence in an attempt to recover any additional evidence related to Noah's murder.

REF: Оба были зарегистрированы полицейским участком Палмдейла, был выдан ордер на обыск жилища, в попытке собрать дополнительные улик

### 4. Дообучение T5
Дообучим маленькую модель [mt5-small](https://huggingface.co/google/mt5-small). В huggingface есть `Seq2SeqTrainer` — он наследуется от Trainer, по гиперпараметрам для запуска почти совпадает с оригиналом. 

Сначала инициализируем модель с токенизатором и напишем функцию токенизации.

In [10]:
from transformers import DataCollatorForSeq2Seq, \
                            Seq2SeqTrainingArguments, \
                            Seq2SeqTrainer, \
                            AutoTokenizer, \
                            AutoModelForSeq2SeqLM


def make_tokenize_fn(tok, max_src=256, max_tgt=256):
    def _fn(batch):
        src = list(batch["src"])
        tgt = batch["tgt"]
        model_inputs = tok(src, max_length=max_src, truncation=True)
        with tok.as_target_tokenizer():
            labels = tok(tgt, max_length=max_tgt, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return _fn


tok = AutoTokenizer.from_pretrained("google/mt5-small")
mdl = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
tok_fn = make_tokenize_fn(tok)
train_ds  = ds["train"].map(tok_fn, batched=True, remove_columns=ds["train"].column_names)
val_ds  = ds["test"].map(tok_fn, batched=True, remove_columns=ds["test"].column_names)

collate = DataCollatorForSeq2Seq(tokenizer=tok, model=mdl)

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

/home/ubuntu/project/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/401 [00:00<?, ? examples/s]

### Задание 4
Допишите гиперпараметры в `Seq2SeqTrainingArguments` для обучения модели. Все параметры, необходимые и специфичные для seq2seq, уже заданы.

In [22]:
# args = Seq2SeqTrainingArguments(
#     'results',
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,
#     num_train_epochs=3,
#     lr_scheduler_type="cosine",
#     learning_rate=1e-5,
#     weight_decay=1e-2,
#     predict_with_generate=True, # генерация результата при валидации
#     )

# Авторское решение
args = Seq2SeqTrainingArguments(
    'results',
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    logging_strategy='epoch',
    logging_steps=1,
    predict_with_generate=True,
    generation_max_length=128,
    report_to="none",
    bf16=True
    )

trainer = Seq2SeqTrainer(model=mdl, 
                         args=args, 
                         train_dataset=train_ds, 
                         eval_dataset=val_ds,
                         processing_class=tok,
                         data_collator=collate)
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.362400,2.702043
2,2.473700,2.614657
3,3.228900,2.576516


TrainOutput(global_step=2403, training_loss=2.6883340201375883, metrics={'train_runtime': 683.2474, 'train_samples_per_second': 7.03, 'train_steps_per_second': 3.517, 'total_flos': 216348469708800.0, 'train_loss': 2.6883340201375883, 'epoch': 3.0})

## 5. Валидация решения
Провалидируем решение. Сначала сгенерируем результат:

In [23]:
res = trainer.predict(val_ds)

Немного подготовим ответы:

In [24]:
preds = res.predictions
preds[preds < 0] = 0 # trainer в качестве пэддинга ставит -100

Финально уберём лишние токены:

In [25]:
hyp = [text.replace('<extra_id_0>', '').strip() 
                  for text in tok.batch_decode(preds, skip_special_tokens=True)]
val_ref = [r["tgt"] for r in ds["test"]]

Посчитаем метрики:

In [26]:
print(compute_bleu_chrf(hyp, val_ref))
# {'bleu': 0.00999980929479024, 'chrf': 11.200873415598856}

{'bleu': 0.041496886158315364, 'chrf': 25.099626866452134}


Метрики ниже метрик модели, предобученной на перевод. Такой результат ожидаем и обоснован — мы обучили модель на небольшом количестве текстов (примерно на 1000 текстов). Чтобы улучшить её, можно попробовать увеличить датасет и размер модели.

In [27]:
for i in range(N):
    print(f"SRC: {val_src[i]}")
    print(f"REF: {val_ref[i]}")
    print(f"HYP: {hyp[i]}")
    print("---\n")

SRC: Taoiseach opens new Irish Consulate in LA

REF: Премьер-министр открыл новое консульство Ирландии в Лос-Анджелесе

HYP: В США присоединится к новому консулию в США
---

SRC: He's been superb.

REF: Он был великолепен.

HYP: В результате он был прекрасным.
---

SRC: "Rising population has become like a second stage cancer.

REF: “Рост населения стал похож на рак второй стадии.

HYP: «Повышение населения стала первой стадии рака.
---

SRC: Perhaps the thing Cramer likes most about the two companies?

REF: Что в этих двух компаниях Крамеру нравится, наверное, больше всего?

HYP: Вообще, если Cramer считает, что его компания может писать о двух компаниях?
---

SRC: The two were booked at Palmdale Sheriff's Station and a search warrant was served at their residence in an attempt to recover any additional evidence related to Noah's murder.

REF: Оба были зарегистрированы полицейским участком Палмдейла, был выдан ордер на обыск жилища, в попытке собрать дополнительные улики, связанные с 